# 02 - Understanding Token Flows for Agent Identities

**Learning objectives**
- Understand different token types for agent identities
- Learn autonomous agent authentication (app-only tokens)
- Explore on-behalf-of (OBO) flow for interactive agents
- Decode and inspect JWT tokens
- Understand token claims and their implications
- Test token usage with Microsoft Graph API

**Prerequisites**
- Completed [01-validate-configuration.ipynb](./01-validate-configuration.ipynb)
- Valid `.env` configuration from `a365.ps1` script
- Understanding of OAuth 2.0 basics

**What you'll learn:**
- **Autonomous Agent Tokens**: How agents authenticate as themselves (client credentials flow)
- **Token Anatomy**: What's inside a JWT and what each claim means
- **Scope vs Permission**: Understanding OAuth scopes and Microsoft Graph permissions
- **Token Lifetime**: How long tokens last and when to refresh
- **Agent Identity vs Blueprint**: How multiple agents share the same blueprint

## Token Flow Overview

Agent identities support three primary authentication patterns:

### 1. Autonomous Agent (App-Only)
The agent acts on its own behalf without a user context.
- **Flow**: OAuth 2.0 Client Credentials
- **Use case**: Background services, automation, scheduled tasks
- **Token type**: Application token (no user claims)
- **Permissions**: Application permissions (e.g., `User.Read.All`)

```
Agent App → Token Endpoint → Access Token → Microsoft Graph
```

### 2. Autonomous Agent User
The agent acts as a dedicated user account (agent has its own mailbox/profile).
- **Flow**: OAuth 2.0 Client Credentials with user context
- **Use case**: Agents that need user-like capabilities (send email, create calendar events)
- **Token type**: Application token with user context
- **Permissions**: Delegated permissions as the agent user

### 3. Interactive Agent (On-Behalf-Of)
The agent acts on behalf of a human user.
- **Flow**: OAuth 2.0 On-Behalf-Of (OBO)
- **Use case**: Chat bots, AI assistants, user-delegated actions
- **Token type**: User token exchanged for agent token
- **Permissions**: Delegated permissions with user consent

```
User → Client App → User Token → Agent API → Exchange for Agent Token → Graph
```

**This notebook focuses on Autonomous Agent (pattern #1) since that's what the `a365.ps1` script configured.**

## Setup: Load Configuration

First, let's load our validated configuration from the previous notebook.

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# Load environment
load_dotenv()

config = {
    "tenant_id": os.getenv("AZURE_TENANT_ID"),
    "client_id": os.getenv("AZURE_CLIENT_ID"),
    "credential_type": os.getenv("AZURE_CLIENT_CREDENTIAL_TYPE"),
    "blueprint_id": os.getenv("AGENT_BLUEPRINT_ID"),
    "identifier_uri": os.getenv("AGENT_BLUEPRINT_IDENTIFIER_URI"),
    "scope": os.getenv("AGENT_BLUEPRINT_SCOPE")
}

# Validate required fields
missing = [k for k, v in config.items() if not v]
if missing:
    raise ValueError(
        f"❌ Missing configuration: {', '.join(missing)}\n"
        "Run 01-validate-configuration.ipynb first."
    )

print("✅ Configuration loaded")
print(f"   Tenant: {config['tenant_id']}")
print(f"   Client: {config['client_id']}")
print(f"   Credential Type: {config['credential_type']}")

## Autonomous Agent Token Acquisition

Let's acquire an app-only token for our agent identity using the client credentials flow.

**What happens:**
1. Agent presents its credentials (secret or certificate) to Azure Entra ID
2. Azure validates the credentials and checks permissions
3. Azure issues an access token with application permissions
4. Agent uses this token to call APIs (Microsoft Graph, custom APIs, etc.)

**Key parameters:**
- `client_id`: Your blueprint's application ID
- `scope`: The resource you want to access (e.g., `https://graph.microsoft.com/.default`)
- `client_credential`: Secret or certificate for authentication

In [ ]:
import msal
from datetime import datetime, timezone

def acquire_agent_token(tenant_id, client_id, credential_type, scope="https://graph.microsoft.com/.default"):
    """
    Acquire an app-only token for the agent identity.
    
    Args:
        tenant_id: Azure AD tenant ID
        client_id: Application (client) ID
        credential_type: 'secret' or 'certificate'
        scope: The resource scope to request
    
    Returns:
        Token result dictionary with 'access_token' and metadata
    """
    authority = f"https://login.microsoftonline.com/{tenant_id}"
    
    # Configure credentials based on type
    if credential_type == "secret":
        client_secret = os.getenv("AZURE_CLIENT_SECRET")
        if not client_secret:
            raise ValueError("AZURE_CLIENT_SECRET not found in environment")
        
        app = msal.ConfidentialClientApplication(
            client_id=client_id,
            client_credential=client_secret,
            authority=authority
        )
    
    elif credential_type == "certificate":
        cert_path = os.getenv("AZURE_CLIENT_CERT_PATH")
        cert_thumbprint = os.getenv("AZURE_CLIENT_CERT_THUMBPRINT")
        
        if not cert_path or not cert_thumbprint:
            raise ValueError("Certificate credentials not found in environment")
        
        with open(cert_path, "r") as f:
            private_key = f.read()
        
        client_credential = {
            "private_key": private_key,
            "thumbprint": cert_thumbprint
        }
        
        app = msal.ConfidentialClientApplication(
            client_id=client_id,
            client_credential=client_credential,
            authority=authority
        )
    else:
        raise ValueError(f"Invalid credential type: {credential_type}")
    
    # Acquire token
    result = app.acquire_token_for_client(scopes=[scope])
    
    if "access_token" not in result:
        error = result.get("error")
        description = result.get("error_description")
        raise Exception(f"Token acquisition failed: {error} - {description}")
    
    return result

# Acquire token for Microsoft Graph
print("🎫 Acquiring autonomous agent token...\n")
token_result = acquire_agent_token(
    tenant_id=config["tenant_id"],
    client_id=config["client_id"],
    credential_type=config["credential_type"],
    scope="https://graph.microsoft.com/.default"
)

access_token = token_result["access_token"]
expires_in = token_result.get("expires_in", 0)

print("✅ Token acquired successfully!\n")
print(f"📊 Token Metadata:")
print(f"   Token Type: {token_result.get('token_type', 'Bearer')}")
print(f"   Expires In: {expires_in} seconds ({expires_in // 60} minutes)")
print(f"   Scope: {token_result.get('scope', 'N/A')}")
print(f"   Token Length: {len(access_token)} characters")
print(f"   Preview: {access_token[:50]}...{access_token[-20:]}")

## Decoding JWT Tokens

Access tokens are JSON Web Tokens (JWT) that contain claims about the identity and permissions.

**JWT Structure:**
```
header.payload.signature
```

Each part is Base64URL-encoded:
- **Header**: Algorithm and token type
- **Payload**: Claims (issuer, audience, subject, expiration, custom claims)
- **Signature**: Cryptographic signature to verify authenticity

Let's decode the token and inspect its claims.

In [ ]:
import jwt
import base64

# Decode token without validation (for inspection purposes)
# In production, always validate tokens properly
decoded = jwt.decode(access_token, options={"verify_signature": False})

# Also decode the header
header = jwt.get_unverified_header(access_token)

print("🏷️  JWT Header:")
print(json.dumps(header, indent=2))
print("\n📋 JWT Payload (Claims):")
print(json.dumps(decoded, indent=2, default=str))

## Understanding Token Claims

Let's examine the key claims in our agent token and understand what they mean.

In [ ]:
from datetime import datetime, timezone

print("🔍 Key Token Claims Analysis:\n")

# Issuer (iss)
issuer = decoded.get("iss")
print(f"1️⃣ Issuer (iss): {issuer}")
print(f"   → Identifies who issued the token (Azure AD for this tenant)\n")

# Audience (aud)
audience = decoded.get("aud")
print(f"2️⃣ Audience (aud): {audience}")
print(f"   → The resource this token is intended for")
print(f"   → Should match the API you're calling (e.g., Microsoft Graph)\n")

# Application ID (appid)
appid = decoded.get("appid")
print(f"3️⃣ Application ID (appid): {appid}")
print(f"   → Your agent blueprint's client ID")
if appid == config["client_id"]:
    print(f"   ✅ Matches configured AZURE_CLIENT_ID\n")
else:
    print(f"   ⚠️  Does not match configured AZURE_CLIENT_ID\n")

# Tenant ID (tid)
tid = decoded.get("tid")
print(f"4️⃣ Tenant ID (tid): {tid}")
print(f"   → Your Azure AD tenant")
if tid == config["tenant_id"]:
    print(f"   ✅ Matches configured AZURE_TENANT_ID\n")
else:
    print(f"   ⚠️  Does not match configured AZURE_TENANT_ID\n")

# Object ID (oid)
oid = decoded.get("oid")
print(f"5️⃣ Object ID (oid): {oid}")
print(f"   → Service principal's object ID in the directory\n")

# Subject (sub)
sub = decoded.get("sub")
print(f"6️⃣ Subject (sub): {sub}")
print(f"   → Unique identifier for the principal (service principal ID)\n")

# Roles (roles)
roles = decoded.get("roles", [])
print(f"7️⃣ Roles: {roles if roles else '(none)'}")
print(f"   → Application permissions granted to this agent")
print(f"   → These are the API permissions you configured in Azure Portal")
if roles:
    for role in roles:
        print(f"      • {role}")
print()

# Identity Provider (idp)
idp = decoded.get("idp")
if idp:
    print(f"8️⃣ Identity Provider (idp): {idp}")
    print(f"   → The identity provider that authenticated the principal\n")

# Token Timestamps
iat = decoded.get("iat")
nbf = decoded.get("nbf")
exp = decoded.get("exp")

print(f"⏰ Token Timestamps:")
if iat:
    iat_time = datetime.fromtimestamp(iat, tz=timezone.utc)
    print(f"   Issued At (iat): {iat_time.isoformat()}")
if nbf:
    nbf_time = datetime.fromtimestamp(nbf, tz=timezone.utc)
    print(f"   Not Before (nbf): {nbf_time.isoformat()}")
if exp:
    exp_time = datetime.fromtimestamp(exp, tz=timezone.utc)
    print(f"   Expires At (exp): {exp_time.isoformat()}")
    
    # Calculate time remaining
    now = datetime.now(timezone.utc)
    time_remaining = exp_time - now
    minutes_remaining = int(time_remaining.total_seconds() / 60)
    print(f"   Time Remaining: {minutes_remaining} minutes")

# Token version
ver = decoded.get("ver")
print(f"\n🔢 Token Version (ver): {ver}")
print(f"   → Version of the Azure AD token format")
print(f"   → v1.0 = classic, v2.0 = converged platform")

## Using the Token to Call Microsoft Graph

Now let's use our token to make API calls to Microsoft Graph.

We'll query:
1. Our own service principal details
2. Organization information (if permissions allow)
3. Application information

In [ ]:
import requests

def call_graph(endpoint, access_token, method="GET", data=None):
    """
    Make an authenticated call to Microsoft Graph API.
    
    Args:
        endpoint: Graph API endpoint (relative or absolute URL)
        access_token: Bearer token
        method: HTTP method (GET, POST, PATCH, DELETE)
        data: Request body for POST/PATCH
    
    Returns:
        Response JSON or raises exception
    """
    # Build full URL if relative
    if not endpoint.startswith("http"):
        endpoint = f"https://graph.microsoft.com/v1.0{endpoint}"
    
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }
    
    response = requests.request(method, endpoint, headers=headers, json=data)
    
    if response.status_code in (200, 201, 204):
        return response.json() if response.content else {}
    else:
        error_detail = response.json() if response.content else {"error": response.text}
        raise Exception(
            f"Graph API call failed ({response.status_code}): "
            f"{error_detail.get('error', {}).get('message', error_detail)}"
        )

# Test 1: Get our service principal
principal_id = os.getenv("AGENT_BLUEPRINT_PRINCIPAL_ID")
print("🔍 Test 1: Querying service principal...\n")

try:
    sp = call_graph(f"/servicePrincipals/{principal_id}", access_token)
    print("✅ Service Principal Retrieved:")
    print(f"   Display Name: {sp.get('displayName')}")
    print(f"   App ID: {sp.get('appId')}")
    print(f"   Object ID: {sp.get('id')}")
    print(f"   Type: {sp.get('servicePrincipalType')}")
    print(f"   Account Enabled: {sp.get('accountEnabled')}\n")
except Exception as e:
    print(f"❌ Failed: {e}\n")

# Test 2: Get organization info (requires Directory.Read.All)
print("🔍 Test 2: Querying organization info...\n")

try:
    org = call_graph("/organization", access_token)
    if org.get("value"):
        org_info = org["value"][0]
        print("✅ Organization Retrieved:")
        print(f"   Display Name: {org_info.get('displayName')}")
        print(f"   Tenant ID: {org_info.get('id')}")
        print(f"   Verified Domains: {len(org_info.get('verifiedDomains', []))}\n")
    else:
        print("⚠️  No organization data returned\n")
except Exception as e:
    print(f"❌ Failed: {e}")
    print("   This may indicate missing Directory.Read.All permission\n")

# Test 3: Get application registration
blueprint_id = config["blueprint_id"]
print("🔍 Test 3: Querying application registration...\n")

try:
    app = call_graph(f"/applications/{blueprint_id}", access_token)
    print("✅ Application Retrieved:")
    print(f"   Display Name: {app.get('displayName')}")
    print(f"   App ID: {app.get('appId')}")
    print(f"   Object ID: {app.get('id')}")
    print(f"   Sign-in Audience: {app.get('signInAudience')}")
    
    # Show identifier URIs
    identifier_uris = app.get('identifierUris', [])
    if identifier_uris:
        print(f"   Identifier URIs:")
        for uri in identifier_uris:
            print(f"      • {uri}")
    
    # Show API scopes
    api_config = app.get('api', {})
    oauth_scopes = api_config.get('oauth2PermissionScopes', [])
    if oauth_scopes:
        print(f"   OAuth Scopes:")
        for scope in oauth_scopes:
            print(f"      • {scope.get('value')} - {scope.get('adminConsentDisplayName')}")
    print()
except Exception as e:
    print(f"❌ Failed: {e}")
    print("   This may indicate missing Application.Read.All permission\n")

## Token Caching and Refresh Strategy

Access tokens have a limited lifetime (typically 60-90 minutes). For long-running applications, you need a token refresh strategy.

**Best practices:**
1. **Cache tokens**: Don't request a new token for every API call
2. **Check expiration**: Monitor the `exp` claim before using cached tokens
3. **Handle 401 errors**: Refresh token when API returns 401 Unauthorized
4. **Use MSAL caching**: MSAL automatically caches and refreshes tokens

Let's implement a simple token manager:

In [ ]:
from datetime import datetime, timezone, timedelta

class TokenManager:
    """
    Simple token manager with caching and automatic refresh.
    """
    
    def __init__(self, tenant_id, client_id, credential_type):
        self.tenant_id = tenant_id
        self.client_id = client_id
        self.credential_type = credential_type
        self._token_cache = {}  # scope -> (token, expiry)
    
    def get_token(self, scope="https://graph.microsoft.com/.default", force_refresh=False):
        """
        Get a cached token or acquire a new one.
        
        Args:
            scope: The resource scope
            force_refresh: Force acquiring a new token
        
        Returns:
            Access token string
        """
        # Check cache
        if not force_refresh and scope in self._token_cache:
            token, expiry = self._token_cache[scope]
            
            # Return cached token if not expired (with 5 minute buffer)
            if datetime.now(timezone.utc) < expiry - timedelta(minutes=5):
                print(f"✅ Using cached token (expires in {(expiry - datetime.now(timezone.utc)).seconds // 60} minutes)")
                return token
        
        # Acquire new token
        print(f"🔄 Acquiring new token for scope: {scope}")
        result = acquire_agent_token(
            tenant_id=self.tenant_id,
            client_id=self.client_id,
            credential_type=self.credential_type,
            scope=scope
        )
        
        token = result["access_token"]
        expires_in = result.get("expires_in", 3600)
        
        # Cache token with expiry
        expiry = datetime.now(timezone.utc) + timedelta(seconds=expires_in)
        self._token_cache[scope] = (token, expiry)
        
        print(f"✅ New token cached (valid for {expires_in // 60} minutes)")
        return token
    
    def clear_cache(self):
        """Clear all cached tokens."""
        self._token_cache.clear()
        print("🗑️  Token cache cleared")

# Create token manager
token_manager = TokenManager(
    tenant_id=config["tenant_id"],
    client_id=config["client_id"],
    credential_type=config["credential_type"]
)

# Test caching
print("\n🧪 Testing token caching:\n")
token1 = token_manager.get_token()
print()
token2 = token_manager.get_token()  # Should use cache
print()
print(f"Tokens identical: {token1 == token2}")

# Test force refresh
print("\n🧪 Testing force refresh:\n")
token3 = token_manager.get_token(force_refresh=True)
print()
print(f"New token identical to cached: {token1 == token3}")

## Understanding Agent Identity vs Blueprint

One of the key concepts in the Agent Identity model is the distinction between **blueprints** and **identities**.

### Agent Identity Blueprint
- **What**: A template/configuration in Azure Entra ID
- **Purpose**: Defines permissions, scopes, and authentication requirements
- **Created**: Once per agent type/role
- **Properties**:
  - Application registration
  - Client credentials (secrets/certificates)
  - OAuth scopes and permissions
  - Identifier URI

### Agent Identity (Instance)
- **What**: A specific instance created from a blueprint
- **Purpose**: Represents a unique agent with its own context and state
- **Created**: Multiple identities can share the same blueprint
- **Properties**:
  - Unique identity ID
  - References the blueprint
  - May have instance-specific metadata
  - Gets its own audit trail

### Analogy
Think of it like a class (blueprint) vs instances (identities):
```python
class CustomerServiceBot:  # Blueprint
    permissions = ["read_tickets", "send_email"]
    
bot_for_customer_123 = CustomerServiceBot()  # Identity 1
bot_for_customer_456 = CustomerServiceBot()  # Identity 2
```

Both bots have the same permissions (from blueprint), but they are distinct identities serving different customers.

### What we created with a365.ps1
The PowerShell script created the **blueprint** (application + service principal + credentials + scopes).

To create actual **agent identities**, you would:
1. Use the Microsoft Graph API to create agent identities
2. Reference the blueprint ID
3. Each identity can then authenticate using the blueprint's credentials
4. The SDK distinguishes agents via the `AgentIdentity` parameter

**In the next notebook**, we'll explore the Microsoft Entra SDK for Agent Identities, which handles creating and managing multiple agent identities from a single blueprint.

## Summary

✅ **What you've learned:**

1. **Token Flows**: Understanding autonomous agent (app-only) authentication
2. **JWT Structure**: Decoding and inspecting token claims
3. **Key Claims**: Issuer, audience, appid, roles, and timestamps
4. **Token Usage**: Making authenticated Microsoft Graph API calls
5. **Token Management**: Caching and refresh strategies
6. **Blueprint vs Identity**: Understanding the template/instance model

**Key takeaways:**
- Agent tokens use OAuth 2.0 client credentials flow (app-only)
- Tokens contain claims that identify the agent and its permissions
- Tokens expire and should be cached/refreshed automatically
- One blueprint can support multiple agent identities
- The Microsoft Entra SDK simplifies token management

**Next steps:**
- **[03-agent-sdk.ipynb](./03-agent-sdk.ipynb)**: Using the Microsoft Entra SDK for Agent Identities
- **[04-interactive-authentication.ipynb](./04-interactive-authentication.ipynb)**: Implementing on-behalf-of flow for user delegation